In [1]:
%load_ext cudf.pandas
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv("/kaggle/input/datasets/mohansacharya/graduate-admissions/Admission_Predict_Ver1.1.csv")

In [3]:
df.isna().sum().reset_index()

,index,0
0,Serial No.,0
1,GRE Score,0
2,TOEFL Score,0
3,University Rating,0
4,SOP,0
5,LOR,0
6,CGPA,0
7,Research,0
8,Chance of Admit,0


In [4]:
print(df.duplicated(subset='Serial No.').sum())
print(df.shape)

0
(500, 9)


In [5]:
df.sample(6)

,Serial No.,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit
167,168,313,102,3,2.0,3.0,8.27,0,0.64
149,150,311,106,2,3.5,3.0,8.26,1,0.79
19,20,303,102,3,3.5,3.0,8.50,0,0.62
95,96,304,100,4,1.5,2.5,7.84,0,0.42
196,197,306,105,2,3.0,2.5,8.26,0,0.73
29,30,310,99,2,1.5,2.0,7.30,0,0.54


In [6]:
df.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Serial No.         500 non-null    int64
 1   GRE Score          500 non-null    int64
 2   TOEFL Score        500 non-null    int64
 3   University Rating  500 non-null    int64
 4   SOP                500 non-null    float64
 5   LOR                500 non-null    float64
 6   CGPA               500 non-null    float64
 7   Research           500 non-null    int64
 8   Chance of Admit    500 non-null    float64
dtypes: float64(4), int64(5)
memory usage: 35.2 KB


In [7]:
df.describe()

,Serial No.,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit
count,500.000000,500.000000,500.000000,500.000000,500.000000,500.00000,500.000000,500.000000,500.00000
mean,250.500000,316.472000,107.192000,3.114000,3.374000,3.48400,8.576440,0.560000,0.72174
std,144.481833,11.295148,6.081868,1.143512,0.991004,0.92545,0.604813,0.496884,0.14114
min,1.000000,290.000000,92.000000,1.000000,1.000000,1.00000,6.800000,0.000000,0.34000
25%,125.750000,308.000000,103.000000,2.000000,2.500000,3.00000,8.127500,0.000000,0.63000
50%,250.500000,317.000000,107.000000,3.000000,3.500000,3.50000,8.560000,1.000000,0.72000
75%,375.250000,325.000000,112.000000,4.000000,4.000000,4.00000,9.040000,1.000000,0.82000
max,500.000000,340.000000,120.000000,5.000000,5.000000,5.00000,9.920000,1.000000,0.97000


In [8]:
# replacing the colomn names
df.columns = df.columns.str.replace(' ','_')
df.sample(3)

,Serial_No.,GRE_Score,TOEFL_Score,University_Rating,SOP,LOR_,CGPA,Research,Chance_of_Admit_
173,174,323,113,4,4.0,4.5,9.23,1,0.89
298,299,330,114,3,4.5,4.5,9.24,1,0.90
66,67,327,114,3,3.0,3.0,9.02,0,0.61


In [9]:
df.columns = df.columns.str.rstrip('_')        # removes any underscore at the end

df.sample(3)

,Serial_No.,GRE_Score,TOEFL_Score,University_Rating,SOP,LOR,CGPA,Research,Chance_of_Admit
431,432,320,112,2,3.5,3.5,8.78,1,0.73
53,54,324,112,4,4.0,2.5,8.10,1,0.72
58,59,300,99,1,3.0,2.0,6.80,1,0.36


In [10]:
# scaling the data

from sklearn.preprocessing import MinMaxScaler         # here we will be using MinMaxScaler, coz here the upper and lower bound is fixed for the different exams, values cant be more or less than that
# standard scaler is used where there is no upper and lower bounds of the data 
scaler = MinMaxScaler()

In [11]:
x = df.drop(columns=['Serial_No.', 'Chance_of_Admit'])
y = df[['Chance_of_Admit']]

x = scaler.fit_transform(x)
x

array([[0.94      , 0.92857143, 0.75      , ..., 0.875     , 0.91346154,
        1.        ],
       [0.68      , 0.53571429, 0.75      , ..., 0.875     , 0.66346154,
        1.        ],
       [0.52      , 0.42857143, 0.5       , ..., 0.625     , 0.38461538,
        1.        ],
       ...,
       [0.8       , 1.        , 1.        , ..., 1.        , 0.88461538,
        1.        ],
       [0.44      , 0.39285714, 0.75      , ..., 1.        , 0.5224359 ,
        0.        ],
       [0.74      , 0.75      , 0.75      , ..., 0.875     , 0.71794872,
        0.        ]])

In [12]:
from sklearn.model_selection import train_test_split as tts

In [13]:
x_train, x_test, y_train, y_test = tts(x, y, test_size=0.2, random_state=42)
print(x_train.shape)
x_train

(400, 7)


array([[0.62      , 0.67857143, 0.5       , ..., 0.75      , 0.65064103,
        1.        ],
       [0.52      , 0.67857143, 0.75      , ..., 1.        , 0.55769231,
        0.        ],
       [0.26      , 0.35714286, 0.5       , ..., 0.5       , 0.54487179,
        0.        ],
       ...,
       [0.24      , 0.25      , 0.        , ..., 0.25      , 0.14423077,
        0.        ],
       [0.38      , 0.46428571, 0.25      , ..., 0.75      , 0.28205128,
        0.        ],
       [0.48      , 0.5       , 0.25      , ..., 0.625     , 0.46474359,
        0.        ]])

### using `optuna` 

In [15]:
!pip install optuna-integration[tfkeras]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 2.5 MB/s eta 0:00:0000:01


In [16]:
import optuna
from optuna.integration import TFKerasPruningCallback

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import models, layers, regularizers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l1, l2

2026-05-17 06:18:48.658018: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778998729.102462      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778998729.226454      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778998730.230951      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778998730.230990      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778998730.230992      57 computation_placer.cc:177] computation placer alr

In [17]:
def build_model(trial):
    # clears all the previous models from the memory, saving the gpu from being crash
    keras.backend.clear_session()

    # here optuna will choose the number of parameters btwn 1 to 5
    n_layers = trial.suggest_int('n_layes', 1,5)

    # here optuna will choose the learning rate
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)

    model = keras.Sequential()

    # takes total nums of cols in x_train
    cols = x_train.shape[1]

    for i in range(n_layers):
        # chooses a random number of neuron for every layer
        num_neurons = trial.suggest_int(f"layer_{i}", 32, 128, step=32)

        if i == 0:
            # putting input_dim
            model.add(layers.Dense(num_neurons, activation='relu', input_dim=cols))
        else:
            model.add(layers.Dense(num_neurons, activation='relu'))

        # putting dropout
        droput_percentage = trial.suggest_float(f"dropout_{i}", 0.1,0.4, step=0.1)
        model.add(layers.Dropout(droput_percentage))

    # final layer of the model
    model.add(layers.Dense(1, activation='linear'))
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='mse',        # using mean squared error for loss
        metrics=['mae']
    )

    history = model.fit(
        x_train, y_train,
        validation_split=0.3,
        # validation_data=(x_test, y_test),          # either use validation_split or validation_data, cant use both
        epochs=50,
        verbose=0,           # keeps the jupyter notebook clean
        batch_size=32,
        callbacks=[TFKerasPruningCallback(trial, 'val_loss')]
    )

    trial.set_user_attr('model', model)
    trial.set_user_attr('history', history.history)

    mae = min(history.history['mae'])
    val_mae = min(history.history['val_mae'])

    loss = min(history.history['loss'])
    val_loss = min(history.history['val_loss'])

    trial.set_user_attr('mae', mae)
    trial.set_user_attr('val_mae', val_mae)
    trial.set_user_attr('loss', loss)
    trial.set_user_attr('val_loss', val_loss)
    
    # Return exactly ONE thing for Optuna to minimize
    # you cant return all the metric, so we just saved all the metric and returned only one value
    return val_loss

In [18]:
# this code tells optuna what to do
study = optuna.create_study(direction='minimize', pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10))

# this code tells optuna how to deal with the model
study.optimize(build_model, n_trials=20)

[I 2026-05-17 06:33:20,420] A new study created in memory with name: no-name-9deb7efe-1621-4bfc-8562-8ae6c3203cae
I0000 00:00:1778999601.150451      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13619 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778999601.155651      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1778999605.412692     397 service.cc:152] XLA service 0x7caf6002b340 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778999605.412746     397 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1778999605.412752     397 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1778999605.951485     397 cuda_

## The bird eye view of what is happending

- The Setup: A tiny database is created in the computer's memory to track the experiment (we can actually see the confirmation log just below the code)
- The Marathon Starts: Optuna will immediately begin building and training 20 different neural networks, one after the other.
- The Live Updates: Underneath that cell, there is a live log printing out. Every time a model finishes, Optuna will print a line saying, "Trial 1 finished with a score of 0.05. Parameters used: 2 layers, 64 neurons..."
- The Silent Executioner: While they are training, Optuna will silently kill off any terrible models early so we don't have to wait for them.
- The Finish Line: After Trial 20 finishes, the cell will stop running. The absolute best neural network out of the 20 is now securely saved inside that study variable, ready for us to pull out!

## The Entire Workflow

When using Optuna to tune a Neural Network, the process is broken down into 4 simple steps. 

### Step 1: The Blueprint (The Function)
You define a function (`build_model`) that tells Keras how to build a neural network. However, instead of hardcoding numbers (like `64` neurons or `0.2` dropout), you leave **blank spaces** using `trial.suggest_...`. 
* **What it does:** It gives Optuna a menu of options. It says, *"You can build the model however you want, but you must pick between 32 and 128 neurons."*

### Step 2: Set the Rules (`create_study`)
Before Optuna can start testing models, it needs to know the rules of the game.
* **What it does:** It creates a database to track the trials. It tells Optuna that the goal is to get the lowest error possible (`direction='minimize'`). It also sets up the "Executioner" (`MedianPruner`), which tells Optuna to immediately kill any model that is performing terribly to save time.

### Step 3: Hit Play (`optimize`)
This is where the Blueprint and the Rules finally connect.
* **What it does:** Optuna takes your Blueprint and runs it 20 times (`n_trials=20`). On every trial, it guesses a different combination of neurons, layers, and learning rates. It remembers every single guess and score in the database created in Step 2.

### Step 4: Claim your Prize (`user_attrs`)
After the 20 trials finish, Optuna looks at its database and says, *"Trial 14 got the lowest error!"*
* **What it does:** Because you cleverly wrote `trial.set_user_attr('model', model)` inside your function, Optuna saved the actual, fully trained neural network from Trial 14 in its memory. You are simply reaching into Optuna's memory, grabbing that specific winning model (`best_model`), and pulling it out so you can make predictions!


```python
import matplotlib.pyplot as plt

# 1. Create the study (Direction is 'minimize' because we want the lowest Mean Squared Error)
study = optuna.create_study(
    direction='minimize', 
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
)

# 2. Start the search! (Let's do 20 trials)
study.optimize(build_model, n_trials=20)

print("\n--- OPTIMIZATION FINISHED ---")
print(f"Best Trial Number: {study.best_trial.number}")
print("Best Parameters:", study.best_params)

# 3. Pull the actual live model and history out of the winning trial
best_model = study.best_trial.user_attrs['model']
best_history = study.best_trial.user_attrs['history']


In [19]:
# printing the best parameters

print(f"BEST TRIAL NUMBER : {study.best_trial.number}")
print()
print(f"BEST PARAMETERS : {study.best_params}")

BEST TRIAL NUMBER : 17

BEST PARAMETERS : {'n_layes': 1, 'learning_rate': 0.005523929278610287, 'layer_0': 96, 'dropout_0': 0.1}


In [27]:
# storing the best model and history in an object

model = study.best_trial.user_attrs['model']
history = study.best_trial.user_attrs['history']

# storing the inputs from the best model
mae = study.best_trial.user_attrs['mae']
val_mae = study.best_trial.user_attrs['val_mae']
loss = study.best_trial.user_attrs['loss']
val_loss = study.best_trial.user_attrs['val_loss']


In [ ]:
history

In [31]:
print(mae)
print(val_mae)
print(loss)
print(val_loss)

0.053171444684267044
0.03821595758199692
0.0045541333965957165
0.0033287247642874718


In [33]:
print(f"Training MAE (Average Error): {mae:.4f}")
print(f"Validation MAE (Average Error): {val_mae:.4f}")
print(f"Training MSE (Loss): {loss:.4f}")
print(f"Validation MSE (Loss): {val_loss:.4f}")

Training MAE (Average Error): 0.0532
Validation MAE (Average Error): 0.0382
Training MSE (Loss): 0.0046
Validation MSE (Loss): 0.0033


In [36]:
# predicting

prediction = model.predict(x_test)

prediction                             # since this is a regression problem there is no need to use argmax()

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 


array([[0.9291431 ],
       [0.81124425],
       [0.5939387 ],
       [0.71388686],
       [0.8281319 ],
       [0.8704438 ],
       [0.50575024],
       [0.63183326],
       [0.82645595],
       [0.8282473 ],
       [0.723757  ],
       [0.7306833 ],
       [0.69410956],
       [0.9522128 ],
       [0.8344877 ],
       [0.5049863 ],
       [0.8664186 ],
       [0.6083847 ],
       [0.5390754 ],
       [0.5691712 ],
       [0.6732179 ],
       [0.52819085],
       [0.71358013],
       [0.7977396 ],
       [0.7935349 ],
       [0.6237203 ],
       [0.97637475],
       [0.8538945 ],
       [0.6499324 ],
       [0.7704921 ],
       [0.5518    ],
       [0.72612286],
       [0.55749285],
       [0.8728194 ],
       [0.67791986],
       [0.7531338 ],
       [0.57716584],
       [0.97925425],
       [0.6657882 ],
       [0.7182226 ],
       [0.95277846],
       [0.5934615 ],
       [0.6697744 ],
       [0.87126446],
       [0.9518343 ],
       [0.58173144],
       [0.96100473],
       [0.850

In [45]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



mae = mean_absolute_error(y_test, prediction)
mse = mean_squared_error(y_test, prediction)
r2 = r2_score(y_test, prediction)


print(f"R2 SCORE (Accuracy-like)  : {np.round(r2, 2) * 100} %")
print(f"MEAN SQUARED ERROR (MSE)  : {np.round(mse, 4)}")
print(f"MEAN ABSOLUTE ERROR (MAE) : {np.round(mae, 4)}")

R2 SCORE (Accuracy-like)  : 81.0 %
MEAN SQUARED ERROR (MSE)  : 0.0039
MEAN ABSOLUTE ERROR (MAE) : 0.0431
